In [22]:
import numpy as np
from numpy.linalg import inv
from scipy.stats import t

# 模拟带有多重共线性的数据
def simulate_data(n, beta, rho=0.95):
    """
    模型: y = beta0 + beta1*x1 + beta2*x2 + beta3*x3 + ε
    其中，为了引入多重共线性，对各个自变量构造共享随机因子。
    """
    # 生成标准正态随机数（不带相关性）
    x_star = np.random.randn(n, len(beta) - 1)
    # 生成公共因子
    common = np.random.randn(n, 1)
    # 利用公式: x = sqrt(1-rho^2)*x* + rho*common 构造相关解释变量
    X_no_intercept = np.sqrt(1 - rho**2) * x_star + rho * common
    # 加入截距项
    X = np.hstack((np.ones((n, 1)), X_no_intercept))
    # 噪声
    sigma = 1.0
    noise = sigma * np.random.randn(n)
    y = X.dot(beta) + noise
    return X, y

# 贝叶斯回归（闭式解）
def bayesian_regression(X, y, theta0, Q0, v0, S0_sq):
    """
    输入：
      X       : 设计矩阵（n×k）
      y       : 因变量
      theta0  : 先验均值 (k×1)
      Q0      : 先验协方差矩阵（未缩放），对应 theta|h ~ N(theta0, h⁻¹Q0)
      v0      : 先验自由度，对应 Gamma 分布参数
      S0_sq   : 先验尺度参数（先验方差的平均值）
    输出：
      theta_star : 后验均值
      Q_star     : 后验协方差（在 h 下的缩放矩阵）
      v_star     : 后验自由度（n+v0）
      S_star_sq  : 后验误差方差（综合了先验和数据）
      theta_hat  : OLS 估计
    """
    n = X.shape[0]
    XtX = X.T.dot(X)
    # 计算 OLS 估计
    theta_hat = inv(XtX).dot(X.T).dot(y)
    
    # 后验精度矩阵：Q_star = (Q0⁻¹ + X'X)⁻¹
    Q0_inv = inv(Q0)
    Q_star = inv(Q0_inv + XtX)
    # 后验均值：theta_star = Q_star * (Q0⁻¹*theta0 + X'X*theta_hat)
    theta_star = Q_star.dot(Q0_inv.dot(theta0) + XtX.dot(theta_hat))
    
    # 后验自由度：v_star = n + v0
    v_star = n + v0
    
    # 计算 OLS 的误差平方和（SSE）
    SSE = np.sum((y - X.dot(theta_hat))**2)
    
    # 根据论文中的公式（见公式(20)），更新尺度参数
    # 这里计算额外项：(theta_hat - theta0)' * [Q0 + (X'X)⁻¹]⁻¹ * (theta_hat - theta0)
    temp = theta_hat - theta0
    term = temp.T.dot(inv(Q0 + inv(XtX))).dot(temp)
    S_star_sq = (v0 * S0_sq + SSE + term) / v_star
    
    # 后验分布为多元 t 分布，位置参数为 theta_star，尺度矩阵为 S_star_sq * Q_star，自由度为 v_star
    return theta_star, Q_star, v_star, S_star_sq, theta_hat

# 示例运行
np.random.seed(42)
n = 100
beta_true = np.array([17, 8.5, 5.0, 2.0])  # 真正的回归系数

# 生成带有高相关性的自变量
X, y = simulate_data(n, beta_true, rho=0.95)

# 先验设定（参考论文中的先验规格）
theta0 = np.array([15, 10, 5.5, 2.5])
Q0 = np.diag([2.4, 6e-7, 0.15, 0.6])  # 注意第二个变量的方差设定很小，体现了强先验信息
v0 = 4
S0_sq = 1.5

theta_star, Q_star, v_star, S_star_sq, theta_hat = bayesian_regression(X, y, theta0, Q0, v0, S0_sq)

print("OLS 估计:", theta_hat)
print("贝叶斯后验均值:", theta_star)
print("后验自由度:", v_star)
print("后验尺度（方差）:", S_star_sq)

# 计算各参数的 95% 可信区间
alpha = 0.05
t_val = t.ppf(1 - alpha / 2, df=v_star)
for i in range(len(theta_star)):
    se = np.sqrt(S_star_sq * Q_star[i, i])
    lower = theta_star[i] - t_val * se
    upper = theta_star[i] + t_val * se
    print(f"参数 {i} 的 95% 可信区间: ({lower:.4f}, {upper:.4f})")
MSE_bayes = np.mean((theta_star - beta_true)**2)
MSE_ols = np.mean((theta_hat - beta_true)**2)
print("\nMSE for Bayesian method:", MSE_bayes)
print("MSE for OLS method:", MSE_ols)

OLS 估计: [16.91798457  8.86801675  5.11796679  1.78286961]
贝叶斯后验均值: [16.8514356   9.99998938  4.65642179  1.23782392]
后验自由度: 104
后验尺度（方差）: 1.3100572626365141
参数 0 的 95% 可信区间: (16.6213, 17.0816)
参数 1 的 95% 可信区间: (9.9982, 10.0017)
参数 2 的 95% 可信区间: (4.2431, 5.0698)
参数 3 的 95% 可信区间: (0.8049, 1.6708)

MSE for Bayesian method: 0.7427494691591482
MSE for OLS method: 0.050806157185676847


In [23]:
import numpy as np
from numpy.linalg import inv
from scipy.stats import t

# 模拟带有多重共线性的数据
def simulate_data(n, beta, rho):
    """
    模型: y = beta0 + beta1*x1 + beta2*x2 + beta3*x3 + ε
    为引入多重共线性，通过公共因子构造相关解释变量。
    """
    x_star = np.random.randn(n, len(beta) - 1)
    common = np.random.randn(n, 1)
    X_no_intercept = np.sqrt(1 - rho**2) * x_star + rho * common
    X = np.hstack((np.ones((n, 1)), X_no_intercept))
    sigma = 1.0
    noise = sigma * np.random.randn(n)
    y = X.dot(beta) + noise
    return X, y

# 贝叶斯回归（闭式解）
def bayesian_regression(X, y, theta0, Q0, v0, S0_sq):
    """
    输入：
      X       : 设计矩阵 (n×k)
      y       : 因变量
      theta0  : 先验均值 (k×1)
      Q0      : 先验协方差矩阵（未缩放），对应 θ|h ~ N(θ0, h⁻¹ Q0)
      v0      : 先验自由度（Gamma 分布参数）
      S0_sq   : 先验尺度参数（先验方差的平均值）
      
    输出：
      theta_star : 后验均值
      Q_star     : 后验协方差（在 h 下的缩放矩阵）
      v_star     : 后验自由度（n+v0）
      S_star_sq  : 后验尺度（误差方差）
      theta_hat  : OLS 估计（作为比较）
    """
    n = X.shape[0]
    XtX = X.T.dot(X)
    theta_hat = inv(XtX).dot(X.T).dot(y)
    
    Q0_inv = inv(Q0)
    Q_star = inv(Q0_inv + XtX)
    theta_star = Q_star.dot(Q0_inv.dot(theta0) + XtX.dot(theta_hat))
    
    v_star = n + v0
    SSE = np.sum((y - X.dot(theta_hat))**2)
    temp = theta_hat - theta0
    term = temp.T.dot(inv(Q0 + inv(XtX))).dot(temp)
    S_star_sq = (v0 * S0_sq + SSE + term) / v_star
    
    return theta_star, Q_star, v_star, S_star_sq, theta_hat

# 示例运行
np.random.seed(42)
n = 60
beta_true = np.array([17, 8.5, 5.0, 2.0])
X, y = simulate_data(n, beta_true, rho=0.95)

# 先验设定（参考论文中的先验规格）
theta0 = np.array([15, 10, 5.5, 2.5])
Q0 = np.diag([2.4, 6e-7, 0.15, 0.6])  # 注意第二个变量的方差设定较小，体现了强先验信息
v0 = 4
S0_sq = 1.5

# 贝叶斯回归
theta_star, Q_star, v_star, S_star_sq, theta_hat = bayesian_regression(X, y, theta0, Q0, v0, S0_sq)

print("OLS 估计:", theta_hat)
print("贝叶斯后验均值:", theta_star)
print("后验自由度:", v_star)
print("后验尺度（方差）:", S_star_sq)

# ----------------------------
# 计算贝叶斯方法下的标准误和可信区间（多元 t 分布）
alpha = 0.05
t_val_bayes = t.ppf(1 - alpha / 2, df=v_star)
bayes_results = []
print("\n【贝叶斯方法】")
for i in range(len(theta_star)):
    se_bayes = np.sqrt(S_star_sq * Q_star[i, i])
    lower_bayes = theta_star[i] - t_val_bayes * se_bayes
    upper_bayes = theta_star[i] + t_val_bayes * se_bayes
    bayes_results.append((theta_star[i], se_bayes, lower_bayes, upper_bayes))
    print(f"参数 {i}: 后验均值 = {theta_star[i]:.4f}, 标准误 = {se_bayes:.4f}, 95% 可信区间 = ({lower_bayes:.4f}, {upper_bayes:.4f})")

# ----------------------------
# 计算 OLS 方法下的标准误和置信区间
n, k = X.shape
s2 = np.sum((y - X.dot(theta_hat))**2) / (n - k)
cov_ols = inv(X.T.dot(X))
ols_se = np.sqrt(s2 * np.diag(cov_ols))
t_val_ols = t.ppf(1 - alpha / 2, df=n - k)
ols_results = []
print("\n【OLS 方法】")
for i in range(len(theta_hat)):
    lower_ols = theta_hat[i] - t_val_ols * ols_se[i]
    upper_ols = theta_hat[i] + t_val_ols * ols_se[i]
    ols_results.append((theta_hat[i], ols_se[i], lower_ols, upper_ols))
    print(f"参数 {i}: OLS 估计 = {theta_hat[i]:.4f}, 标准误 = {ols_se[i]:.4f}, 95% 置信区间 = ({lower_ols:.4f}, {upper_ols:.4f})")
# 计算均方误差（MSE），以评估两种方法的估计精度
MSE_bayes = np.mean((theta_star - beta_true)**2)
MSE_ols = np.mean((theta_hat - beta_true)**2)
print("\nMSE for Bayesian method:", MSE_bayes)
print("MSE for OLS method:", MSE_ols)

OLS 估计: [16.98168172  8.57135196  4.62855492  2.09032811]
贝叶斯后验均值: [16.92244506  9.9999912   4.42668366  1.08161861]
后验自由度: 64
后验尺度（方差）: 1.517393511712979

【贝叶斯方法】
参数 0: 后验均值 = 16.9224, 标准误 = 0.1587, 95% 可信区间 = (16.6055, 17.2394)
参数 1: 后验均值 = 10.0000, 标准误 = 0.0010, 95% 可信区间 = (9.9981, 10.0019)
参数 2: 后验均值 = 4.4267, 标准误 = 0.2885, 95% 可信区间 = (3.8503, 5.0030)
参数 3: 后验均值 = 1.0816, 标准误 = 0.2724, 95% 可信区间 = (0.5374, 1.6258)

【OLS 方法】
参数 0: OLS 估计 = 16.9817, 标准误 = 0.1361, 95% 置信区间 = (16.7090, 17.2543)
参数 1: OLS 估计 = 8.5714, 标准误 = 0.3928, 95% 置信区间 = (7.7844, 9.3583)
参数 2: OLS 估计 = 4.6286, 标准误 = 0.3764, 95% 置信区间 = (3.8744, 5.3827)
参数 3: OLS 估计 = 2.0903, 标准误 = 0.3410, 95% 置信区间 = (1.4073, 2.7733)

MSE for Bayesian method: 0.8570260944724792
MSE for OLS method: 0.03788931864599672
